# 09 — Grid Frequency & PCC Voltage Safety Signals

This notebook visualises the two new **safety-critical signals** added to C2G-Bench:

1. **Grid frequency** — modelled by a simplified swing equation.
   - Tracking deficit drives supply-demand imbalance → frequency deviation.
   - Thresholds: UFLS at ±0.5 Hz, dead-band ±0.2 Hz for reward penalty.

2. **PCC voltage** — steady-state Thévenin voltage drop at the Point of Common Coupling.
   - Heavier facility load → larger voltage drop.
   - ANSI C84.1 Range A: [0.95, 1.05] pu; relay trip at 0.90 pu.

Both signals are now part of the 16-D observation space (indices 14–15) and contribute
to the reward function and episode termination conditions.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from c2g_env.simulators.macro_grid import MacroGridSignal
from c2g_env.simulators.electrical import DatacenterElectrical
from c2g_env.env_low_level import C2GFastEnv

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (12, 4),
                      "axes.grid": True, "grid.alpha": 0.3})
print("Imports OK")


## 1. Swing-Equation Frequency Response

We drive the grid model with a known tracking deficit (MW) and observe how
the swing equation integrates frequency deviation over time.


In [ ]:
# Create grid model (60 Hz, no added noise for clarity)
grid = MacroGridSignal(energy_dir="data/processed/energy", zone="NYC",
                       committed_mw=20.0, seed=42)
grid._f_noise_std = 0.0  # deterministic for visualisation

dt = 5.0  # seconds per tick
n_ticks = 360  # 30 minutes

# Scenario: 5 MW deficit for 2 min, then 0 for 8 min, then -3 MW for 5 min, then 0
deficits = np.zeros(n_ticks)
deficits[0:24]    = 5.0    # 0–2 min: excess demand
deficits[120:180] = -3.0   # 10–15 min: excess supply

freqs = np.zeros(n_ticks)
for i in range(n_ticks):
    grid._step_frequency(deficits[i])
    freqs[i] = grid.f_grid

time_min = np.arange(n_ticks) * dt / 60.0

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

ax1.fill_between(time_min, deficits, alpha=0.4, color="C1", label="Tracking deficit (MW)")
ax1.axhline(0, color="gray", lw=0.5)
ax1.set_ylabel("Deficit [MW]")
ax1.legend(loc="upper right")
ax1.set_title("Grid Frequency Swing-Equation Response")

ax2.plot(time_min, freqs, color="C0", lw=1.5, label="Grid frequency")
ax2.axhline(60.0, color="gray", ls="--", lw=0.8, label="Nominal (60 Hz)")
ax2.axhline(59.8, color="orange", ls=":", lw=0.8, label="Dead-band (±0.2 Hz)")
ax2.axhline(60.2, color="orange", ls=":", lw=0.8)
ax2.axhline(59.5, color="red", ls="--", lw=0.8, label="UFLS (59.5 Hz)")
ax2.axhline(60.5, color="red", ls="--", lw=0.8, label="OFGT (60.5 Hz)")
ax2.set_ylabel("Frequency [Hz]")
ax2.set_xlabel("Time [minutes]")
ax2.legend(loc="lower right", fontsize=8, ncol=2)
ax2.set_ylim(59.3, 60.7)

plt.tight_layout()
plt.show()


## 2. Frequency Sensitivity to Inertia Constant (H)

The inertia constant H determines how quickly the system responds to
disturbances. Lower H (fewer synchronous generators) → faster frequency swings.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

for H in [3.0, 6.0, 10.0]:
    g = MacroGridSignal(energy_dir="data/processed/energy", zone="NYC",
                        committed_mw=20.0, seed=42)
    g._f_noise_std = 0.0
    g._H_inertia = H
    fs = []
    for i in range(n_ticks):
        g._step_frequency(deficits[i])
        fs.append(g.f_grid)
    ax.plot(time_min, fs, lw=1.5, label=f"H = {H:.0f} s")

ax.axhline(60.0, color="gray", ls="--", lw=0.8)
ax.axhspan(59.8, 60.2, alpha=0.08, color="green", label="Dead-band (no penalty)")
ax.set_ylabel("Frequency [Hz]")
ax.set_xlabel("Time [minutes]")
ax.set_title("Frequency Response vs. System Inertia")
ax.legend(fontsize=9)
ax.set_ylim(59.3, 60.7)
plt.tight_layout()
plt.show()


## 3. PCC Voltage Drop vs. Facility Load

The Thévenin equivalent model computes steady-state voltage at the Point
of Common Coupling (PCC). Heavier load → larger IR/IX drop → lower voltage.


In [ ]:
elec = DatacenterElectrical()

utils = np.linspace(0.0, 1.0, 50)
voltages = []
drops = []
for u in utils:
    r = elec.step(u, u, u * 15.0, u * 8.0)
    voltages.append(r["v_pcc_pu"])
    drops.append(r["v_drop_pu"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(utils, voltages, "C0-", lw=2)
ax1.axhline(1.0, color="gray", ls="--", lw=0.8, label="Nominal (1.0 pu)")
ax1.axhspan(0.95, 1.05, alpha=0.08, color="green", label="ANSI C84.1 Range A")
ax1.axhline(0.90, color="red", ls="--", lw=0.8, label="UV relay trip (0.90 pu)")
ax1.set_xlabel("Server Utilisation")
ax1.set_ylabel("PCC Voltage [pu]")
ax1.set_title("PCC Voltage vs. Load")
ax1.legend(fontsize=8)
ax1.set_ylim(0.85, 1.05)

ax2.plot(utils, drops, "C1-", lw=2)
ax2.set_xlabel("Server Utilisation")
ax2.set_ylabel("Voltage Drop [pu]")
ax2.set_title("Voltage Drop vs. Load")
ax2.set_ylim(0, max(drops) * 1.2)

plt.tight_layout()
plt.show()


## 4. Full Environment Episode with Frequency & Voltage

Run a 5-minute episode (60 ticks) with random actions and observe how
frequency and voltage evolve alongside temperature and reward.


In [ ]:
env = C2GFastEnv(scenario="default")
obs, _ = env.reset(seed=42)

ticks = 360  # 30 minutes
data = {"freq_hz": [], "freq_dev": [], "v_pcc": [], "reward": [],
        "temp_A": [], "tracking_err": [], "freq_pen": [], "volt_pen": [],
        "thermal_fault": [], "freq_fault": [], "volt_fault": []}

for t in range(ticks):
    action = env.action_space.sample()
    obs, rew, terminated, truncated, info = env.step(action)
    data["freq_hz"].append(info["f_grid_hz"])
    data["freq_dev"].append(info["freq_dev_hz"])
    data["v_pcc"].append(info["v_pcc_pu"])
    data["reward"].append(rew)
    data["temp_A"].append(info["temp_A"])
    data["tracking_err"].append(info["tracking_err_kw"])
    data["freq_pen"].append(info["freq_penalty"])
    data["volt_pen"].append(info["volt_penalty"])
    data["thermal_fault"].append(info["thermal_fault"])
    data["freq_fault"].append(info["freq_fault"])
    data["volt_fault"].append(info["voltage_fault"])
    if terminated or truncated:
        print(f"Episode ended at tick {t}: terminated={terminated}, truncated={truncated}")
        break

time_s = np.arange(len(data["freq_hz"])) * 5.0 / 60.0
print(f"Ran {len(data['freq_hz'])} ticks ({len(data['freq_hz'])*5/60:.1f} min)")


In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)

# 1. Grid frequency
ax = axes[0]
ax.plot(time_s, data["freq_hz"], "C0-", lw=1)
f_nom = data["freq_hz"][0]  # approximate
ax.axhline(f_nom, color="gray", ls="--", lw=0.8)
ax.axhline(f_nom - 0.2, color="orange", ls=":", lw=0.8)
ax.axhline(f_nom + 0.2, color="orange", ls=":", lw=0.8)
ax.axhline(f_nom - 0.5, color="red", ls="--", lw=0.8)
ax.axhline(f_nom + 0.5, color="red", ls="--", lw=0.8)
ax.set_ylabel("Frequency [Hz]")
ax.set_title("Episode Trajectory — Random Agent")

# 2. PCC voltage
ax = axes[1]
ax.plot(time_s, data["v_pcc"], "C2-", lw=1)
ax.axhline(1.0, color="gray", ls="--", lw=0.8)
ax.axhspan(0.95, 1.05, alpha=0.06, color="green")
ax.axhline(0.90, color="red", ls="--", lw=0.8)
ax.set_ylabel("PCC Voltage [pu]")

# 3. Safety penalties
ax = axes[2]
ax.plot(time_s, data["freq_pen"], "C1-", lw=1, label="Freq penalty")
ax.plot(time_s, data["volt_pen"], "C3-", lw=1, label="Volt penalty")
ax.set_ylabel("Penalty")
ax.legend(fontsize=8)

# 4. Reward
ax = axes[3]
ax.plot(time_s, data["reward"], "C4-", lw=1)
ax.set_ylabel("Reward")
ax.set_xlabel("Time [minutes]")

plt.tight_layout()
plt.show()


## 5. Fault Flag Summary

Count how many ticks triggered each fault type during the random episode.


In [ ]:
n = len(data["thermal_fault"])
thermal_count = sum(data["thermal_fault"])
freq_count    = sum(data["freq_fault"])
volt_count    = sum(data["volt_fault"])

print(f"Episode length: {n} ticks ({n*5/60:.1f} min)")
print(f"Thermal faults: {thermal_count}")
print(f"Frequency faults: {freq_count}")
print(f"Voltage faults: {volt_count}")
print(f"Total termination events: {thermal_count + freq_count + volt_count}")


## 6. 50 Hz vs 60 Hz Market Comparison

Compare frequency dynamics between a US market (60 Hz) and European market (50 Hz).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (market, zone, f_nom_label) in zip(axes, [
    ("nyiso_nyc", "NYC", "60 Hz (US)"),
    ("entso_de",  "DE",  "50 Hz (EU)")
]):
    g = MacroGridSignal(energy_dir="data/processed/energy", zone=zone,
                        committed_mw=20.0, seed=42, market=market)
    g._f_noise_std = 0.002
    fs = []
    for i in range(n_ticks):
        g._step_frequency(deficits[i])
        fs.append(g.f_grid)
    f_nom = g.f_nom
    ax.plot(time_min, fs, "C0-", lw=1.5)
    ax.axhline(f_nom, color="gray", ls="--", lw=0.8)
    ax.axhspan(f_nom - 0.2, f_nom + 0.2, alpha=0.08, color="green")
    ax.axhline(f_nom - 0.5, color="red", ls="--", lw=0.8)
    ax.axhline(f_nom + 0.5, color="red", ls="--", lw=0.8)
    ax.set_title(f"{f_nom_label} — {market}")
    ax.set_xlabel("Time [min]")
    ax.set_ylabel("Frequency [Hz]")
    ax.set_ylim(f_nom - 0.7, f_nom + 0.7)

plt.tight_layout()
plt.show()


## Summary

| Signal | Obs Index | Range | Penalty Dead-band | Termination |
|--------|-----------|-------|-------------------|-------------|
| `freq_dev_norm` | 14 | [-1, 1] | ±0.2 Hz | \|Δf\| > 0.5 Hz (UFLS/OFGT) |
| `v_pcc_pu` | 15 | [0, 1.1] | [0.95, 1.05] pu | v < 0.90 pu (UV relay) |

These signals are critical for designing **safety controllers** (constrained RL,
shielding, Lagrangian methods) that maintain grid stability while optimising
data center operations.
